## Phase 1 - Defining the Agents

### 1. Import AgentPy

In [5]:
#  import the library
!pip install agentpy --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
  Using cached agentpy-0.1.5-py3-none-any.whl.metadata (3.3 kB)
  Using cached salib-1.5.2-py3-none-any.whl.metadata (11 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
Using cached agentpy-0.1.5-py3-none-any.whl (53 kB)
Using cached salib-1.5.2-py3-none-any.whl (780 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 2.6 MB/s eta 0:00:00a 0:00:01
Using cached dill-0.4.1-py3-none-any.whl (120 kB)


In [1]:
import agentpy as ap


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/lib/python3/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/lib/python3/dist-packages/traitlets/config/application.py", line 982, in launch_instance
    app.start()
  File "/usr/lib/python3/dist-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/usr/lib/python3/dist-packages/tornado/platform/asyncio.py", line 205, in s

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/lib/python3/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/lib/python3/dist-packages/traitlets/config/application.py", line 982, in launch_instance
    app.start()
  File "/usr/lib/python3/dist-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/usr/lib/python3/dist-packages/tornado/platform/asyncio.py", line 205, in s

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



### 2.The MachineAgent

In [2]:
class MachineAgent(ap.Agent):

    # The setup method acts as our constructor in AgentPy
    def setup(self, capability="Drilling"):
        # Save the machine's capability
        self.capability = capability

        # Initialize an empty list to act as our message inbox
        self.inbox = []

        # Print a startup message confirming the agent's ID and capability
        print(f"Machine Agent {self.id} is ready. Registered capability: {self.capability}")

### 3. The OrderAgent

In [3]:
class OrderAgent(ap.Agent):

    def setup(self, required_operation="Drilling"):
        # Save what operation this order needs
        self.required_operation = required_operation

        # Initialize its own empty inbox
        self.inbox = []

        # Print a startup message
        print(f"Order Agent {self.id} is ready. Required operation: {self.required_operation}")

#### 4. Testing the Agents

In [4]:
# 1. Create a blank base model to house our agents
dummy_model = ap.Model()

# 2. Spawn one MachineAgent and pass the capability argument
machine1 = MachineAgent(dummy_model, capability="Drilling")

# 3. Spawn one OrderAgent and pass the required_operation argument
order1 = OrderAgent(dummy_model, required_operation="Drilling")

Machine Agent 1 is ready. Registered capability: Drilling
Order Agent 2 is ready. Required operation: Drilling


## Phase 2 - Building the Model Environment

### This is where AgentPy really simplifies things compared to JADE. Instead of relying on a complex Directory Facilitator (the Yellow Pages) to discover other agents, AgentPy uses a centralized Model to orchestrate everything. The Model acts as the environment that holds all the agents together. This means that any agent can easily access any other agent through the Model, without needing to go through a separate discovery process. The Model provides a straightforward way for agents to find and interact with each other, making the implementation much simpler and more intuitive.

### 5. The Model Environment

In [5]:
class ManufacturingModel(ap.Model):

    def setup(self):
        # 1. Instantiate agents using ap.AgentList
        # This creates lists containing 1 agent of each type, passing the required arguments
        self.machines = ap.AgentList(self, 1, MachineAgent, capability="Drilling")
        self.orders = ap.AgentList(self, 1, OrderAgent, required_operation="Drilling")

        # 2. Service Discovery: Give the OrderAgent its "Contact List"
        # We grab the first (and only) order agent at index 0 and hand it the list of machines
        self.orders[0].target_machines = self.machines

        print("\n--- Model Setup Complete ---")

# 3. Execution Block
if __name__ == '__main__':
    # Instantiate the model
    model = ManufacturingModel()

    # Run the setup to spawn the agents and trigger their print statements
    model.setup()

Machine Agent 1 is ready. Registered capability: Drilling
Order Agent 2 is ready. Required operation: Drilling

--- Model Setup Complete ---


## Phase 3 - Enabling Agent Communication

### By using simple state variables (like self.state), we can easily control whether an agent does something every single tick (like listening) or just once (like sending an initial request).

#### 6. Updated MachineAgent with a step method to listen for requests

In [6]:
class MachineAgent(ap.Agent):

    def setup(self, capability="Drilling"):
        self.capability = capability
        self.inbox = []
        print(f"Machine Agent {self.id} is ready. Registered capability: {self.capability}")

    # --- PHASE 3: The Step Method ---
    def step(self):
        # Because there is no state check here, this will print on EVERY tick
        print(f"Machine Agent {self.id} is waiting for work...")

#### 7. Updated OrderAgent with a state variable to control when it sends its request

In [7]:
class OrderAgent(ap.Agent):

    def setup(self, required_operation="Drilling"):
        self.required_operation = required_operation
        self.inbox = []

        # --- PHASE 3: State Tracking ---
        self.state = 'seeking_machine'

        print(f"Order Agent {self.id} is ready. Required operation: {self.required_operation}")

    # --- PHASE 3: The Step Method ---
    def step(self):
        # This acts like a JADE OneShotBehaviour because we change the state immediately after!
        if self.state == 'seeking_machine':
            print(f"Order Agent {self.id} is looking for machines...")

            # Change the state so this block never runs again
            self.state = 'waiting_for_bids'

### 8. Updated Model and Execution

#### To actually see the time steps in action, we need to update our model to tell the agents to take their steps, and then use AgentPy's built-in model.run() function instead of just calling setup().

In [8]:
class ManufacturingModel(ap.Model):

    def setup(self):
        self.machines = ap.AgentList(self, 1, MachineAgent, capability="Drilling")
        self.orders = ap.AgentList(self, 1, OrderAgent, required_operation="Drilling")

        # Service Discovery configuration
        self.orders[0].target_machines = self.machines
        print("\n--- Model Setup Complete ---\n")

    # --- PHASE 3: Model Step ---
    def step(self):
        # On every tick of the simulation, tell our agent lists to run their step() methods
        self.machines.step()
        self.orders.step()

# Execution Block
if __name__ == '__main__':
    model = ManufacturingModel()

    # Run the model for exactly 2 time steps to see our state logic work!
    # (We hide the default AgentPy output by storing it in 'results')
    results = model.run(steps=2, display=False)

Machine Agent 1 is ready. Registered capability: Drilling
Order Agent 2 is ready. Required operation: Drilling

--- Model Setup Complete ---

Machine Agent 1 is waiting for work...
Order Agent 2 is looking for machines...
Machine Agent 1 is waiting for work...


## Phase 4 - Agent Communication (Simulated Messaging)

### 9.Updated MachineAgent (The Receiver)

#### We are replacing the basic print statement with a loop that checks the self.inbox. It reads the dictionary, prints the result, and deletes the message so it isn't read twice.

In [9]:
class MachineAgent(ap.Agent):

    def setup(self, capability="Drilling"):
        self.capability = capability
        self.inbox = []
        print(f"Machine Agent {self.id} is ready. Registered capability: {self.capability}")

    # --- PHASE 4: Reading the Inbox ---
    def step(self):
        # We iterate over a copy of the list [:] so we can safely .remove() items from the real list
        for msg in self.inbox.copy():

            # Check the intent (performative) of the message
            if msg["performative"] == "CFP":
                sender_id = msg['sender'].id
                print(f">>> Machine {self.id} received a job request from Order {sender_id}.")
                print(f"    Message content: {msg['content']}")

            # Clear the message from the inbox after reading it
            self.inbox.remove(msg)

### 10. Updated OrderAgent (The Sender)

#### We are updating the seeking_machine state to construct our Python dictionary message and append it directly to the target machines.

In [10]:
class OrderAgent(ap.Agent):

    def setup(self, required_operation="Drilling"):
        self.required_operation = required_operation
        self.inbox = []
        self.state = 'seeking_machine'
        print(f"Order Agent {self.id} is ready. Required operation: {self.required_operation}")

    # --- PHASE 4: Sending the Message ---
    def step(self):
        if self.state == 'seeking_machine':
            print(f"<<< Order {self.id} is broadcasting Call for Proposals (CFP)...")

            # 1. Construct the simulated FIPA-ACL message
            message = {
                "performative": "CFP",
                "sender": self,  # We pass a reference to ourselves so they can reply!
                "content": "Job_Request"
            }

            # 2. Drop the message into the inbox of every capable machine
            for machine in self.target_machines:
                machine.inbox.append(message)

            # 3. Change state to wait for replies
            self.state = 'waiting_for_bids'

### 11. Updated Model and Execution

#### To really see the broadcasting work, let's bump the number of machines up to two in our model setup, and ensure we call model.run(steps=2).

In [11]:
class ManufacturingModel(ap.Model):

    def setup(self):
        # Let's spawn 2 machines to prove the OrderAgent can broadcast!
        self.machines = ap.AgentList(self, 2, MachineAgent, capability="Drilling")
        self.orders = ap.AgentList(self, 1, OrderAgent, required_operation="Drilling")

        self.orders[0].target_machines = self.machines
        print("\n--- Model Setup Complete ---\n")

    def step(self):
        # Order dictates execution sequence per tick.
        # We run orders first so they send the message, then machines so they read it in the same tick.
        self.orders.step()
        self.machines.step()

# Execution Block
if __name__ == '__main__':
    model = ManufacturingModel()

    # Run the model for 2 steps
    results = model.run(steps=2, display=False)

Machine Agent 1 is ready. Registered capability: Drilling
Machine Agent 2 is ready. Registered capability: Drilling
Order Agent 3 is ready. Required operation: Drilling

--- Model Setup Complete ---

<<< Order 3 is broadcasting Call for Proposals (CFP)...
>>> Machine 1 received a job request from Order 3.
    Message content: Job_Request
>>> Machine 2 received a job request from Order 3.
    Message content: Job_Request


## Phase 5- The Contract Net Protocol Challenge

### 12. Final MachineAgent (The Bidder)

#### We need to import random at the top of this cell. The machine will now read the CFP, check if it has the right tools for the job, and if so, send a PROPOSE message back to the sender's inbox. We also add logic to read the final ACCEPT or REJECT messages.

In [12]:
import random
import agentpy as ap

class MachineAgent(ap.Agent):

    def setup(self, capability="Drilling"):
        self.capability = capability
        self.inbox = []
        print(f"Machine {self.id} is ready. Capability: {self.capability}")

    # --- PHASE 5: The Bidding Logic ---
    def step(self):
        for msg in self.inbox.copy():

            # 1. Received a Job Request (CFP)
            if msg["performative"] == "CFP":
                required_op = msg["content"]

                # Check if we can actually do the job!
                if self.capability == required_op:
                    processing_time = random.randint(10, 50)
                    print(f"<<< Machine {self.id} ({self.capability}) proposing time: {processing_time}")

                    # Send a bid back to the OrderAgent
                    reply = {
                        "performative": "PROPOSE",
                        "sender": self,
                        "content": processing_time
                    }
                    msg["sender"].inbox.append(reply)
                else:
                    # If we can't do it, we just ignore the CFP
                    print(f"--- Machine {self.id} ignoring request (Needs {required_op}, but I do {self.capability})")

            # 2. We won the bid!
            elif msg["performative"] == "ACCEPT_PROPOSAL":
                print(f"WINNER! >>> Machine {self.id} got the job! Starting manufacturing...")

            # 3. We lost the bid
            elif msg["performative"] == "REJECT_PROPOSAL":
                print(f"LOSER   >>> Machine {self.id} lost the bid. Back to waiting.")

            # Clear message after reading
            self.inbox.remove(msg)

### 13. Final OrderAgent (The Auctioneer)

#### The Order Agent will now broadcast its required operation. Once it enters the waiting_for_bids state, it will collect all PROPOSE messages, find the lowest time, and hand out the accept/reject messages.

In [13]:
class OrderAgent(ap.Agent):

    def setup(self, required_operation="Drilling"):
        self.required_operation = required_operation
        self.inbox = []
        self.state = 'seeking_machine'
        print(f"Order {self.id} is ready. Looking for: {self.required_operation}")

    # --- PHASE 5: Awarding the Contract ---
    def step(self):
        # State 1: Broadcast the CFP
        if self.state == 'seeking_machine':
            print(f"\n<<< Order {self.id} broadcasting CFP for {self.required_operation}...")
            message = {
                "performative": "CFP",
                "sender": self,
                # Pass what we need as the content so machines can check their capabilities
                "content": self.required_operation
            }
            for machine in self.target_machines:
                machine.inbox.append(message)

            self.state = 'waiting_for_bids'

        # State 2: Evaluate the Bids
        elif self.state == 'waiting_for_bids':
            # Filter our inbox for only 'PROPOSE' messages
            proposals = [msg for msg in self.inbox if msg["performative"] == "PROPOSE"]

            if len(proposals) > 0:
                print(f"\n--- Order {self.id} evaluating {len(proposals)} bids ---")

                # Find the proposal with the lowest processing time (the 'content' field)
                best_proposal = min(proposals, key=lambda x: x["content"])

                # Loop through the proposals to send out Accept/Reject messages
                for p in proposals:
                    reply = {"sender": self}

                    if p == best_proposal:
                        reply["performative"] = "ACCEPT_PROPOSAL"
                        reply["content"] = "You are hired!"
                        print(f"<<< Awarding job to Machine {p['sender'].id} (Time: {p['content']})")
                    else:
                        reply["performative"] = "REJECT_PROPOSAL"
                        reply["content"] = "Too slow!"

                    # Drop the reply into the machine's inbox
                    p["sender"].inbox.append(reply)

                # Clear evaluated proposals from our inbox
                for p in proposals:
                    self.inbox.remove(p)

                # Change state so we don't evaluate again!
                self.state = 'assigned'
                print("--- Negotiation Complete ---\n")

### 14. Final Model and Execution

In [16]:
class ManufacturingModel(ap.Model):

    def setup(self):
        # Let's create a mix of machines!
        # Two Drilling machines to compete, and one Milling machine to test the capability check.
        drilling_machines = ap.AgentList(self, 2, MachineAgent, capability="Drilling")
        milling_machines = ap.AgentList(self, 1, MachineAgent, capability="Milling")

        # Combine them into one big list of 3 machines
        self.machines = drilling_machines + milling_machines

        self.orders = ap.AgentList(self, 1, OrderAgent, required_operation="Drilling")

        # Give the OrderAgent the full contact list
        self.orders[0].target_machines = self.machines
        print("\n--- Model Setup Complete ---")

    def step(self):
        # Execution order matters! Orders act, then machines react in the same tick.
        self.orders.step()
        self.machines.step()

# Execution Block
if __name__ == '__main__':
    model = ManufacturingModel()

    # Run the model for 5 steps to allow the full negotiation back-and-forth
    results = model.run(steps=5, display=False)

Machine 1 is ready. Capability: Drilling
Machine 2 is ready. Capability: Drilling
Machine 3 is ready. Capability: Milling
Order 4 is ready. Looking for: Drilling

--- Model Setup Complete ---

<<< Order 4 broadcasting CFP for Drilling...
<<< Machine 1 (Drilling) proposing time: 50
<<< Machine 2 (Drilling) proposing time: 47
--- Machine 3 ignoring request (Needs Drilling, but I do Milling)

--- Order 4 evaluating 2 bids ---
<<< Awarding job to Machine 2 (Time: 47)
--- Negotiation Complete ---

LOSER   >>> Machine 1 lost the bid. Back to waiting.
WINNER! >>> Machine 2 got the job! Starting manufacturing...
